In [1]:
# ============================================================
# STEP 5 - CELL 1
# SETUP
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

import json
import re
import gc
import os
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/step5")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("STEP 5 - FAISS + RAG RETRIEVAL")
print("=" * 70)

print("Input :", INPUT_DIR)
print("Output:", OUTPUT_DIR)

STEP 5 - FAISS + RAG RETRIEVAL
Input : /kaggle/input
Output: /kaggle/working/step5


In [2]:
# ============================================================
# STEP 5 - CELL 2
# CHECK FAISS
# ============================================================

try:

    import faiss

except ImportError:

    !pip install -q faiss-cpu

    import faiss


print("FAISS version:", faiss.__version__)
print("✅ FAISS ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 74.4 MB/s eta 0:00:00:00:0100:01
FAISS version: 1.15.1
✅ FAISS ready


In [3]:
# ============================================================
# STEP 5 - CELL 3
# DISCOVER INPUT FILES
# ============================================================

def find_file(filename):

    matches = list(
        INPUT_DIR.rglob(filename)
    )

    if not matches:

        raise FileNotFoundError(
            f"{filename} not found"
        )

    if len(matches) > 1:

        print(
            f"⚠️ Multiple {filename} found."
        )

        print(
            "Using:",
            matches[0]
        )

    return matches[0]


files = {

    "clean_products":
        find_file("clean_products.csv"),

    "clean_reviews":
        find_file("clean_reviews.csv"),

    "product_embeddings":
        find_file("product_embeddings.npy"),

    "product_metadata":
        find_file("product_embedding_metadata.csv"),

    "review_embeddings":
        find_file("review_embeddings.npy"),

    "review_metadata":
        find_file("review_embedding_metadata.csv"),

    "embedding_config":
        find_file("embedding_config.json")
}


for name, path in files.items():

    print(
        f"{name:<22}: {path}"
    )

clean_products        : /kaggle/input/datasets/trnquangtriu/5dataset/clean_products.csv
clean_reviews         : /kaggle/input/datasets/trnquangtriu/5dataset/clean_reviews.csv
product_embeddings    : /kaggle/input/datasets/trnquangtriu/5dataset/product_embeddings.npy
product_metadata      : /kaggle/input/datasets/trnquangtriu/5dataset/product_embedding_metadata.csv
review_embeddings     : /kaggle/input/datasets/trnquangtriu/5dataset/review_embeddings.npy
review_metadata       : /kaggle/input/datasets/trnquangtriu/5dataset/review_embedding_metadata.csv
embedding_config      : /kaggle/input/datasets/trnquangtriu/5dataset/embedding_config.json


# Load Embeddings + Metadata

In [4]:
# ============================================================
# STEP 5 - CELL 4
# LOAD EMBEDDINGS & METADATA
# ============================================================

print("=" * 70)
print("LOAD EMBEDDING ARTIFACTS")
print("=" * 70)


product_embeddings = np.load(
    files["product_embeddings"],
    mmap_mode="r"
)

review_embeddings = np.load(
    files["review_embeddings"],
    mmap_mode="r"
)


product_metadata = pd.read_csv(
    files["product_metadata"],
    low_memory=False
)

review_metadata = pd.read_csv(
    files["review_metadata"],
    low_memory=False
)


with open(
    files["embedding_config"],
    "r",
    encoding="utf-8"
) as f:

    embedding_config = json.load(f)


print(
    "Product embeddings:",
    product_embeddings.shape
)

print(
    "Product metadata  :",
    product_metadata.shape
)

print(
    "Review embeddings :",
    review_embeddings.shape
)

print(
    "Review metadata   :",
    review_metadata.shape
)

print("\nEmbedding config:")

print(
    json.dumps(
        embedding_config,
        indent=2,
        ensure_ascii=False
    )
)

LOAD EMBEDDING ARTIFACTS
Product embeddings: (164926, 384)
Product metadata  : (164926, 24)
Review embeddings : (714798, 384)
Review metadata   : (714798, 9)

Embedding config:
{
  "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "embedding_dimension": 384,
  "max_sequence_length": 128,
  "normalized_embeddings": true,
  "similarity": "cosine_via_inner_product",
  "review_embedding_text": "clean_review",
  "product_embedding_text": "product_text",
  "review_vectors": 714798,
  "product_vectors": 164926
}


# Consistency Check trước FAISS

In [5]:
# ============================================================
# STEP 5 - CELL 5
# PRE-FAISS CONSISTENCY CHECK
# ============================================================

print("=" * 70)
print("PRE-FAISS CONSISTENCY CHECK")
print("=" * 70)


DIM = int(
    embedding_config[
        "embedding_dimension"
    ]
)


checks = {

    "product_rows":
        product_embeddings.shape[0]
        == len(product_metadata),

    "review_rows":
        review_embeddings.shape[0]
        == len(review_metadata),

    "product_dimension":
        product_embeddings.shape[1]
        == DIM,

    "review_dimension":
        review_embeddings.shape[1]
        == DIM,

    "product_embedding_id":
        np.array_equal(
            product_metadata[
                "product_embedding_id"
            ].values,
            np.arange(
                len(product_metadata)
            )
        ),

    "review_embedding_id":
        np.array_equal(
            review_metadata[
                "review_embedding_id"
            ].values,
            np.arange(
                len(review_metadata)
            )
        )
}


for name, status in checks.items():

    print(
        ("✅" if status else "❌"),
        f"{name:<25}:",
        status
    )


assert all(
    checks.values()
)

print(
    "\n✅ Embedding artifacts are consistent"
)


PRE-FAISS CONSISTENCY CHECK
✅ product_rows             : True
✅ review_rows              : True
✅ product_dimension        : True
✅ review_dimension         : True
✅ product_embedding_id     : True
✅ review_embedding_id      : True

✅ Embedding artifacts are consistent


# Build Product FAISS

In [6]:
# ============================================================
# STEP 5 - CELL 6
# BUILD PRODUCT FAISS INDEX
# ============================================================

print("=" * 70)
print("BUILD PRODUCT FAISS INDEX")
print("=" * 70)


product_index = faiss.IndexFlatIP(
    DIM
)


PRODUCT_CHUNK = 50_000

for start in range(
    0,
    len(product_embeddings),
    PRODUCT_CHUNK
):

    end = min(
        start + PRODUCT_CHUNK,
        len(product_embeddings)
    )

    vectors = np.asarray(
        product_embeddings[start:end],
        dtype=np.float32
    )

    product_index.add(
        vectors
    )

    print(
        f"Added {end:,} / "
        f"{len(product_embeddings):,}"
    )


print("\nIndex vectors:", product_index.ntotal)
print("Dimension    :", product_index.d)


assert (
    product_index.ntotal
    == len(product_metadata)
)

print("\n✅ Product FAISS index built")

BUILD PRODUCT FAISS INDEX
Added 50,000 / 164,926
Added 100,000 / 164,926
Added 150,000 / 164,926
Added 164,926 / 164,926

Index vectors: 164926
Dimension    : 384

✅ Product FAISS index built


# Build Review FAISS

In [7]:
# ============================================================
# STEP 5 - CELL 7
# BUILD REVIEW FAISS INDEX
# ============================================================

print("=" * 70)
print("BUILD REVIEW FAISS INDEX")
print("=" * 70)


review_index = faiss.IndexFlatIP(
    DIM
)


REVIEW_CHUNK = 50_000

for start in range(
    0,
    len(review_embeddings),
    REVIEW_CHUNK
):

    end = min(
        start + REVIEW_CHUNK,
        len(review_embeddings)
    )

    vectors = np.asarray(
        review_embeddings[start:end],
        dtype=np.float32
    )

    review_index.add(
        vectors
    )

    print(
        f"Added {end:,} / "
        f"{len(review_embeddings):,}"
    )

    del vectors
    gc.collect()


print("\nIndex vectors:", review_index.ntotal)
print("Dimension    :", review_index.d)


assert (
    review_index.ntotal
    == len(review_metadata)
)

print("\n✅ Review FAISS index built")

BUILD REVIEW FAISS INDEX
Added 50,000 / 714,798
Added 100,000 / 714,798
Added 150,000 / 714,798
Added 200,000 / 714,798
Added 250,000 / 714,798
Added 300,000 / 714,798
Added 350,000 / 714,798
Added 400,000 / 714,798
Added 450,000 / 714,798
Added 500,000 / 714,798
Added 550,000 / 714,798
Added 600,000 / 714,798
Added 650,000 / 714,798
Added 700,000 / 714,798
Added 714,798 / 714,798

Index vectors: 714798
Dimension    : 384

✅ Review FAISS index built


# Save 2 FAISS Index

In [8]:
# ============================================================
# STEP 5 - CELL 8
# SAVE FAISS INDEXES
# ============================================================

product_index_path = (
    OUTPUT_DIR /
    "product_faiss.index"
)

review_index_path = (
    OUTPUT_DIR /
    "review_faiss.index"
)


faiss.write_index(
    product_index,
    str(product_index_path)
)

faiss.write_index(
    review_index,
    str(review_index_path)
)


print(
    "Product index:",
    product_index_path
)

print(
    "Review index :",
    review_index_path
)


for path in [
    product_index_path,
    review_index_path
]:

    size_mb = (
        path.stat().st_size /
        1024**2
    )

    print(
        f"{path.name:<25}"
        f"{size_mb:>10.2f} MB"
    )

Product index: /kaggle/working/step5/product_faiss.index
Review index : /kaggle/working/step5/review_faiss.index
product_faiss.index          241.59 MB
review_faiss.index          1047.07 MB


# Load Model để Query

In [14]:
# ============================================================
# STEP 5 - CELL 9
# LOAD QUERY MODEL + QUERY ENCODER
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np
import torch

print("=" * 70)
print("LOAD QUERY EMBEDDING MODEL")
print("=" * 70)

# ------------------------------------------------------------
# 1. Get model name from Step 4 config
# ------------------------------------------------------------

MODEL_NAME = embedding_config.get(
    "model_name",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Model :", MODEL_NAME)
print("Device:", DEVICE)


# ------------------------------------------------------------
# 2. Load model
# ------------------------------------------------------------

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

MODEL_DIM = (
    model.get_sentence_embedding_dimension()
)

print("Dimension:", MODEL_DIM)


# ------------------------------------------------------------
# 3. Validate dimension
# ------------------------------------------------------------

assert MODEL_DIM == DIM, (
    f"Model dimension {MODEL_DIM} "
    f"!= FAISS dimension {DIM}"
)


# ------------------------------------------------------------
# 4. Query encoder
# ------------------------------------------------------------

def encode_query(query):

    if query is None:
        raise ValueError("Query cannot be None")

    query = str(query).strip()

    if query == "":
        raise ValueError("Query cannot be empty")

    vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    vector = np.asarray(
        vector,
        dtype=np.float32
    )

    return vector


# ------------------------------------------------------------
# 5. Sanity test
# ------------------------------------------------------------

test_query = encode_query(
    "tai nghe bluetooth"
)

print("\nTest query shape:", test_query.shape)
print(
    "Vector norm:",
    round(
        float(
            np.linalg.norm(
                test_query[0]
            )
        ),
        6
    )
)

assert test_query.shape == (1, DIM)

assert np.isfinite(
    test_query
).all()

print("\n✅ Query model + encoder ready")

LOAD QUERY EMBEDDING MODEL
Model : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dimension: 384

Test query shape: (1, 384)
Vector norm: 1.0

✅ Query model + encoder ready


# Hàm encode query

In [15]:
# ============================================================
# STEP 5 - CELL 10
# QUERY ENCODER
# ============================================================

def encode_query(query):

    vector = model.encode(

        [str(query)],

        convert_to_numpy=True,

        normalize_embeddings=True,

        show_progress_bar=False
    )

    return vector.astype(
        np.float32
    )


test_query = encode_query(
    "tai nghe bluetooth"
)

print(
    "Shape:",
    test_query.shape
)

print(
    "Norm:",
    np.linalg.norm(
        test_query[0]
    )
)

Shape: (1, 384)
Norm: 1.0


# Product Candidate Retrieval

In [16]:
# ============================================================
# STEP 5 - CELL 11
# PRODUCT CANDIDATE RETRIEVAL
# ============================================================

def retrieve_product_candidates(
    query,
    candidate_k=100
):

    candidate_k = min(
        candidate_k,
        product_index.ntotal
    )

    query_vector = encode_query(
        query
    )

    scores, indices = (
        product_index.search(
            query_vector,
            candidate_k
        )
    )

    indices = indices[0]
    scores = scores[0]

    valid = (
        indices >= 0
    )

    indices = indices[valid]
    scores = scores[valid]


    result = (
        product_metadata
        .iloc[indices]
        .copy()
        .reset_index(drop=True)
    )

    result[
        "semantic_score"
    ] = scores

    return result


candidates = (
    retrieve_product_candidates(
        "tai nghe bluetooth",
        candidate_k=100
    )
)


print(
    "Candidates:",
    len(candidates)
)

display(
    candidates[
        [
            c
            for c in [
                "product_id",
                "source",
                "product_name",
                "category",
                "price",
                "currency",
                "average_rating",
                "semantic_score"
            ]
            if c in candidates.columns
        ]
    ].head(10)
)

Candidates: 100


,product_id,source,product_name,category,price,currency,average_rating,semantic_score
0,amazon_B06XX7QKL2,amazon,"Urbanears Stadion in-Ear Active Wireless Bluetooth Headset, Trail (04091870), Blue",All Electronics,32.99,USD,3.8,0.750951
1,amazon_B074PCB3XB,amazon,SYLVANIA SP328-Turquoise Portable Bluetooth Speaker,All Electronics,23.99,USD,3.8,0.742384
2,amazon_B083YZ61VN,amazon,JBL Duet Mini 2 Bluetooth Wireless In-Ear Headphones Blue,All Electronics,50.49,USD,4.0,0.740766
3,amazon_B0C5PC4J6W,amazon,"BlueParrott S450-XT Voice-Controlled Bluetooth Headset – Industry Leading Sound with Long Wireless Range, Extreme Comfort and Up to 24 Hours of Ta...",Cell Phones & Accessories,172.70,USD,4.2,0.732338
4,amazon_B0B7W59NZL,amazon,"Besign BE-RCA Pro Long Range Bluetooth Adapter for Bluetooth Music Streaming, Black",All Electronics,24.99,USD,4.4,0.729202
5,amazon_B07CDZD8B7,amazon,"Marshall Major III Bluetooth Wireless On-Ear Headphones, Black - New",All Electronics,136.92,USD,4.5,0.726985
6,amazon_B00WSLYAV4,amazon,Sennheiser MM 550-X Wireless Bluetooth Travel Headphones,Home Audio & Theater,399.99,USD,3.7,0.725476
7,amazon_B07V9ZMDX7,amazon,BackBeat PRO 5100 True Wireless Bluetooth Earbuds,Cell Phones & Accessories,34.99,USD,3.7,0.725197
8,amazon_B07Y55ZDRC,amazon,"Bluedio V5.0 Bluetooth Headphones Over Ear, Wireless Bluetooth On-Ear Stereo Earphones Noise Cancelling, Soft Memory-Protein Earmuffs, w/Mic (Black)",Computers,29.99,USD,4.1,0.722842
9,amazon_B09PMLBD17,amazon,Bluedio S6 Sport Earhooks Earphones Wireless Earbuds Bluetooth Headphones 42hrs Play with LED Display Charging Case Built-in Mic Headset for Sport...,All Electronics,16.99,USD,3.8,0.722766


# Structured Product Filters

In [17]:
# ============================================================
# STEP 5 - CELL 12
# STRUCTURED PRODUCT FILTERS
# ============================================================

def apply_product_filters(
    candidates,
    source=None,
    category=None,
    min_price=None,
    max_price=None,
    currency=None,
    min_rating=None
):

    result = candidates.copy()


    # --------------------------------------------------------
    # Source
    # --------------------------------------------------------

    if source is not None:

        result = result[
            result["source"]
            .astype(str)
            .str.lower()
            .eq(
                str(source).lower()
            )
        ]


    # --------------------------------------------------------
    # Currency
    # --------------------------------------------------------

    if (
        currency is not None
        and "currency" in result.columns
    ):

        result = result[
            result["currency"]
            .astype(str)
            .str.upper()
            .eq(
                str(currency).upper()
            )
        ]


    # --------------------------------------------------------
    # Category
    # --------------------------------------------------------

    if (
        category is not None
        and "category" in result.columns
    ):

        result = result[
            result["category"]
            .fillna("")
            .astype(str)
            .str.contains(
                str(category),
                case=False,
                regex=False
            )
        ]


    # --------------------------------------------------------
    # Price
    # --------------------------------------------------------

    if (
        "price"
        in result.columns
    ):

        numeric_price = pd.to_numeric(
            result["price"],
            errors="coerce"
        )

        if min_price is not None:

            result = result[
                numeric_price >= min_price
            ]

        if max_price is not None:

            result = result[
                numeric_price <= max_price
            ]


    # --------------------------------------------------------
    # Rating
    # --------------------------------------------------------

    if (
        min_rating is not None
        and "average_rating"
        in result.columns
    ):

        rating = pd.to_numeric(
            result[
                "average_rating"
            ],
            errors="coerce"
        )

        result = result[
            rating >= min_rating
        ]


    return (
        result
        .reset_index(drop=True)
    )

# Metadata-aware Reranking

In [18]:
# ============================================================
# STEP 5 - CELL 13
# METADATA-AWARE RERANKING
# ============================================================

def minmax(series):

    series = pd.to_numeric(
        series,
        errors="coerce"
    )

    valid = series.dropna()

    if len(valid) == 0:

        return pd.Series(
            0.0,
            index=series.index
        )

    min_v = valid.min()
    max_v = valid.max()

    if max_v == min_v:

        result = pd.Series(
            0.0,
            index=series.index
        )

        result.loc[
            series.notna()
        ] = 1.0

        return result

    return (
        (series - min_v)
        /
        (max_v - min_v)
    ).fillna(0.0)


def rerank_products(
    candidates
):

    result = candidates.copy()

    # Semantic score is already cosine-like,
    # but normalize within candidate set for combination.

    result[
        "semantic_norm"
    ] = minmax(
        result[
            "semantic_score"
        ]
    )


    # --------------------------------------------------------
    # Rating
    # --------------------------------------------------------

    if (
        "average_rating"
        in result.columns
    ):

        rating = pd.to_numeric(
            result[
                "average_rating"
            ],
            errors="coerce"
        )

        # rating already approximately 0-5
        result[
            "rating_norm"
        ] = (
            rating / 5.0
        ).clip(
            0,
            1
        ).fillna(0)

    else:

        result[
            "rating_norm"
        ] = 0.0


    # --------------------------------------------------------
    # Popularity / review count
    # --------------------------------------------------------

    if (
        "review_count"
        in result.columns
    ):

        review_count = pd.to_numeric(
            result[
                "review_count"
            ],
            errors="coerce"
        ).fillna(0)

        result[
            "popularity_norm"
        ] = minmax(
            np.log1p(
                review_count
            )
        )

    else:

        result[
            "popularity_norm"
        ] = 0.0


    # --------------------------------------------------------
    # Final retrieval score
    # --------------------------------------------------------

    result[
        "retrieval_score"
    ] = (

        0.80
        * result[
            "semantic_norm"
        ]

        +

        0.15
        * result[
            "rating_norm"
        ]

        +

        0.05
        * result[
            "popularity_norm"
        ]
    )


    return (
        result
        .sort_values(
            "retrieval_score",
            ascending=False
        )
        .reset_index(drop=True)
    )

# Unified Product Search

In [19]:
# ============================================================
# STEP 5 - CELL 14
# UNIFIED PRODUCT SEARCH
# ============================================================

def search_products(
    query,
    top_k=10,
    candidate_k=200,
    source=None,
    category=None,
    min_price=None,
    max_price=None,
    currency=None,
    min_rating=None
):

    # --------------------------------------------------------
    # Stage 1 - Semantic candidate retrieval
    # --------------------------------------------------------

    candidates = (
        retrieve_product_candidates(
            query,
            candidate_k=candidate_k
        )
    )


    # --------------------------------------------------------
    # Stage 2 - Structured filters
    # --------------------------------------------------------

    filtered = (
        apply_product_filters(

            candidates,

            source=source,
            category=category,

            min_price=min_price,
            max_price=max_price,

            currency=currency,

            min_rating=min_rating
        )
    )


    if len(filtered) == 0:

        return filtered


    # --------------------------------------------------------
    # Stage 3 - Metadata reranking
    # --------------------------------------------------------

    ranked = rerank_products(
        filtered
    )


    return (
        ranked
        .head(top_k)
        .reset_index(drop=True)
    )

# Test Product Retrieval

In [20]:
# ============================================================
# STEP 5 - CELL 15
# PRODUCT RETRIEVAL TEST
# ============================================================

tests = [

    {
        "query":
            "tai nghe bluetooth",

        "top_k":
            5
    },

    {
        "query":
            "wireless bluetooth headphones",

        "top_k":
            5,

        "currency":
            "USD",

        "max_price":
            50
    },

    {
        "query":
            "giày thể thao nam",

        "top_k":
            5,

        "source":
            "tiki",

        "currency":
            "VND",

        "max_price":
            500000
    },

    {
        "query":
            "digital camera photography",

        "top_k":
            5,

        "source":
            "amazon"
    }
]


show_cols = [

    "product_id",
    "source",
    "product_name",
    "category",

    "price",
    "currency",

    "average_rating",
    "review_count",

    "semantic_score",
    "retrieval_score"
]


for params in tests:

    print(
        "\n" + "=" * 70
    )

    print(
        "QUERY:",
        params["query"]
    )

    print(
        "=" * 70
    )

    result = search_products(
        **params
    )

    cols = [
        c
        for c in show_cols
        if c in result.columns
    ]

    display(
        result[cols]
    )


QUERY: tai nghe bluetooth


,product_id,source,product_name,category,price,currency,average_rating,review_count,semantic_score,retrieval_score
0,amazon_B06XX7QKL2,amazon,"Urbanears Stadion in-Ear Active Wireless Bluetooth Headset, Trail (04091870), Blue",All Electronics,32.99,USD,3.8,218,0.750951,0.938963
1,amazon_B074PCB3XB,amazon,SYLVANIA SP328-Turquoise Portable Bluetooth Speaker,All Electronics,23.99,USD,3.8,168,0.742384,0.843530
2,amazon_B083YZ61VN,amazon,JBL Duet Mini 2 Bluetooth Wireless In-Ear Headphones Blue,All Electronics,50.49,USD,4.0,246,0.740766,0.833492
3,amazon_B0C5PC4J6W,amazon,"BlueParrott S450-XT Voice-Controlled Bluetooth Headset – Industry Leading Sound with Long Wireless Range, Extreme Comfort and Up to 24 Hours of Ta...",Cell Phones & Accessories,172.70,USD,4.2,1326,0.732338,0.754576
4,amazon_B0B7W59NZL,amazon,"Besign BE-RCA Pro Long Range Bluetooth Adapter for Bluetooth Music Streaming, Black",All Electronics,24.99,USD,4.4,207,0.729202,0.717500



QUERY: wireless bluetooth headphones


,product_id,source,product_name,category,price,currency,average_rating,review_count,semantic_score,retrieval_score
0,amazon_B0C1ZHJRGB,amazon,Wireless Earbuds Bluetooth Headphones 48hrs Play Back Sport Earphones with LED Display Over-Ear Buds with Earhooks Built-in Mic Headset for Workou...,All Electronics,29.99,USD,4.5,29423,0.856290,0.981973
1,amazon_B0BWK6B2T4,amazon,"Sony WH-CH520 Wireless Headphones Bluetooth On-Ear Headset with Microphone, Blue New",All Electronics,38.00,USD,4.5,491,0.846779,0.862851
2,amazon_B09L3X7KPH,amazon,"LOBKIN Bluetooth Headphone Over Ear, Stereo Wireless Headset with Microphone, Foldable Wireless and Wired Headphones with TF Card MP3 Mode and FM ...",All Electronics,37.99,USD,4.3,2423,0.845866,0.857729
3,amazon_B081PS7YHC,amazon,"Wireless Earbuds Bluetooth Headset with Microphone Sports Headset Touch TWS Stereo Headset in-Ear Bluetooth 50H Playtime Charging Box, Sports, Exe...",All Electronics,19.99,USD,3.1,36,0.847188,0.808848
4,amazon_B091TNX39Z,amazon,"Bluetooth Headphones Wireless Neckband Bluetooth 5.0 Headset with 20 H Playtime, 10 mm Drivers, Magnetic Earbuds, Crystal-Clear Voice and Noise Ca...",All Electronics,15.99,USD,3.4,373,0.842774,0.788665



QUERY: giày thể thao nam


,product_id,source,product_name,category,price,currency,average_rating,review_count,semantic_score,retrieval_score
0,tiki_16532307,tiki,"Lót giày thể thao biết thở thoáng khí và êm chân dùng cho giày chạy bộ, giày tập gym, giày đi bộ, giày sneaker, giày lười, giày búp bê - Loại 3 lớ...",Thời Trang,43000.0,VND,4.7,24,0.787496,0.977324
1,tiki_144183418,tiki,Giày Thể Thao Nam Đế Cao TT01 Trắng,Giày thể thao nam cổ cao,209000.0,VND,4.7,10,0.760030,0.752483
2,tiki_181488092,tiki,Giày thể thao sneaker nam nữ đế cao thời trang,Giày thể thao cổ cao,175000.0,VND,5.0,1,0.756521,0.714710
3,tiki_8193848,tiki,Lót Tăng Chiều Cao Cho Giày Thể Thao Nam Yellow Sport 2 Cm,Thời Trang,39864.0,VND,4.3,7,0.756392,0.708343
4,tiki_171644838,tiki,Lót Giày Thể Thao Êm Chân Thoáng Khí Chống Hôi Chân Khi Sử Dụng Giày Thường Xuyên -LG03,Miếng lót giày nam,18000.0,VND,5.0,1,0.752467,0.682890



QUERY: digital camera photography


,product_id,source,product_name,category,price,currency,average_rating,review_count,semantic_score,retrieval_score
0,amazon_B002W1U1RW,amazon,Digital Concepts Digital Camera & Binocular 27379,Camera & Photo,89.99,USD,3.1,148,0.722723,0.912376
1,amazon_B083DLZT3L,amazon,Neewer 8.5x10ft Backdrop Stand Support Continuous Lighting Kit for Photography Video Shooting,Camera & Photo,219.49,USD,4.6,293,0.703001,0.786468
2,amazon_B000A7BIDQ,amazon,Digital Concepts 37mm TeleandWide Camcorder Lens Kit (KIT1337),Camera & Photo,25.99,USD,3.8,26,0.703207,0.746549
3,amazon_B0C49HY4SV,amazon,"Saneen Digital Camera, 4K Cameras for Photography, 44MP Compact Point and Shoot Photography Cameras for Teens, Kids, Elder, Beginners, 16X Digital...",Camera & Photo,58.99,USD,4.1,48,0.694046,0.678246
4,amazon_B0BNN99R3W,amazon,"ISHARE Digital Camera for Beginners - 30MP 1080P 18X Digital Zoom 2.8”LCD Screen, Point and Shoot Camera with 2X Batteries and 32G SD Card, Ideal ...",Camera & Photo,49.99,USD,4.0,28,0.689999,0.635236


# Product-Type Lexical Reranking

In [21]:
# ============================================================
# STEP 5 - CELL 16
# QUERY-PRODUCT LEXICAL SIGNAL
# ============================================================

STOPWORDS = {

    # English
    "for", "the", "a", "an", "and",
    "with", "of", "to",

    # Vietnamese
    "cho", "và", "của", "với",
    "một", "các", "loại"
}


def tokenize_simple(text):

    tokens = re.findall(
        r"\w+",
        str(text).lower(),
        flags=re.UNICODE
    )

    return {

        token
        for token in tokens

        if (
            len(token) >= 2
            and token not in STOPWORDS
        )
    }


def lexical_overlap(
    query,
    product_name
):

    q = tokenize_simple(
        query
    )

    p = tokenize_simple(
        product_name
    )

    if not q:

        return 0.0

    return (
        len(
            q.intersection(p)
        )
        /
        len(q)
    )

# FINAL PRODUCT RERANKER


In [22]:
# ============================================================
# STEP 5 - CELL 17
# FINAL PRODUCT RERANKER
# ============================================================

def rerank_products_final(
    query,
    candidates
):

    result = candidates.copy()


    result[
        "semantic_norm"
    ] = minmax(
        result[
            "semantic_score"
        ]
    )


    # Rating
    if (
        "average_rating"
        in result.columns
    ):

        rating = pd.to_numeric(
            result[
                "average_rating"
            ],
            errors="coerce"
        )

        result[
            "rating_norm"
        ] = (
            rating / 5
        ).clip(
            0,
            1
        ).fillna(0)

    else:

        result[
            "rating_norm"
        ] = 0.0


    # Popularity
    if (
        "review_count"
        in result.columns
    ):

        review_count = pd.to_numeric(
            result[
                "review_count"
            ],
            errors="coerce"
        ).fillna(0)

        result[
            "popularity_norm"
        ] = minmax(
            np.log1p(
                review_count
            )
        )

    else:

        result[
            "popularity_norm"
        ] = 0.0


    # Lexical
    result[
        "lexical_score"
    ] = result[
        "product_name"
    ].fillna("").apply(

        lambda name:
        lexical_overlap(
            query,
            name
        )
    )


    result[
        "retrieval_score"
    ] = (

        0.70
        * result[
            "semantic_norm"
        ]

        +

        0.15
        * result[
            "lexical_score"
        ]

        +

        0.10
        * result[
            "rating_norm"
        ]

        +

        0.05
        * result[
            "popularity_norm"
        ]
    )


    return (
        result
        .sort_values(
            "retrieval_score",
            ascending=False
        )
        .reset_index(drop=True)
    )

# FINAL PRODUCT SEARCH


In [23]:
# ============================================================
# STEP 5 - CELL 18
# FINAL PRODUCT SEARCH
# ============================================================

def search_products_final(
    query,
    top_k=10,
    candidate_k=200,
    source=None,
    category=None,
    min_price=None,
    max_price=None,
    currency=None,
    min_rating=None
):

    candidates = (
        retrieve_product_candidates(
            query,
            candidate_k
        )
    )


    candidates = (
        apply_product_filters(

            candidates,

            source=source,
            category=category,

            min_price=min_price,
            max_price=max_price,

            currency=currency,

            min_rating=min_rating
        )
    )


    if len(candidates) == 0:

        return candidates


    ranked = (
        rerank_products_final(
            query,
            candidates
        )
    )


    return (
        ranked
        .head(top_k)
        .reset_index(drop=True)
    )

# FINAL PRODUCT SEARCH


In [24]:
# ============================================================
# STEP 5 - CELL 18
# FINAL PRODUCT SEARCH
# ============================================================

def search_products_final(
    query,
    top_k=10,
    candidate_k=200,
    source=None,
    category=None,
    min_price=None,
    max_price=None,
    currency=None,
    min_rating=None
):

    candidates = (
        retrieve_product_candidates(
            query,
            candidate_k
        )
    )


    candidates = (
        apply_product_filters(

            candidates,

            source=source,
            category=category,

            min_price=min_price,
            max_price=max_price,

            currency=currency,

            min_rating=min_rating
        )
    )


    if len(candidates) == 0:

        return candidates


    ranked = (
        rerank_products_final(
            query,
            candidates
        )
    )


    return (
        ranked
        .head(top_k)
        .reset_index(drop=True)
    )

# Test lại các query khó

In [25]:
# ============================================================
# STEP 5 - CELL 19
# FINAL PRODUCT RETRIEVAL TEST
# ============================================================

queries = [

    "tai nghe bluetooth",

    "giày thể thao nam",

    "digital camera",

    "camera for photography",

    "wireless headphones"
]


for query in queries:

    print(
        "\n" + "=" * 70
    )

    print(
        "QUERY:",
        query
    )

    print(
        "=" * 70
    )

    result = (
        search_products_final(
            query,
            top_k=5,
            candidate_k=200
        )
    )

    cols = [

        "product_id",
        "source",
        "product_name",

        "price",
        "currency",

        "average_rating",

        "semantic_score",
        "lexical_score",
        "retrieval_score"
    ]

    cols = [
        c
        for c in cols
        if c in result.columns
    ]

    display(
        result[cols]
    )


QUERY: tai nghe bluetooth


,product_id,source,product_name,price,currency,average_rating,semantic_score,lexical_score,retrieval_score
0,amazon_B06XX7QKL2,amazon,"Urbanears Stadion in-Ear Active Wireless Bluetooth Headset, Trail (04091870), Blue",32.99,USD,3.8,0.750951,0.333333,0.850963
1,amazon_B074PCB3XB,amazon,SYLVANIA SP328-Turquoise Portable Bluetooth Speaker,23.99,USD,3.8,0.742384,0.333333,0.767309
2,amazon_B083YZ61VN,amazon,JBL Duet Mini 2 Bluetooth Wireless In-Ear Headphones Blue,50.49,USD,4.0,0.740766,0.333333,0.757495
3,amazon_B0C5PC4J6W,amazon,"BlueParrott S450-XT Voice-Controlled Bluetooth Headset – Industry Leading Sound with Long Wireless Range, Extreme Comfort and Up to 24 Hours of Ta...",172.70,USD,4.2,0.732338,0.333333,0.688168
4,amazon_B0B7W59NZL,amazon,"Besign BE-RCA Pro Long Range Bluetooth Adapter for Bluetooth Music Streaming, Black",24.99,USD,4.4,0.729202,0.333333,0.653403



QUERY: giày thể thao nam


,product_id,source,product_name,price,currency,average_rating,semantic_score,lexical_score,retrieval_score
0,tiki_16532307,tiki,"Lót giày thể thao biết thở thoáng khí và êm chân dùng cho giày chạy bộ, giày tập gym, giày đi bộ, giày sneaker, giày lười, giày búp bê - Loại 3 lớ...",43000.0,VND,4.7,0.787496,0.75,0.942824
1,tiki_144183418,tiki,Giày Thể Thao Nam Đế Cao TT01 Trắng,209000.0,VND,4.7,0.760030,1.00,0.782430
2,tiki_8193848,tiki,Lót Tăng Chiều Cao Cho Giày Thể Thao Nam Yellow Sport 2 Cm,39864.0,VND,4.3,0.756392,1.00,0.745858
3,tiki_181488092,tiki,Giày thể thao sneaker nam nữ đế cao thời trang,175000.0,VND,5.0,0.756521,1.00,0.745099
4,tiki_171644838,tiki,Lót Giày Thể Thao Êm Chân Thoáng Khí Chống Hôi Chân Khi Sử Dụng Giày Thường Xuyên -LG03,18000.0,VND,5.0,0.752467,0.75,0.679757



QUERY: digital camera


,product_id,source,product_name,price,currency,average_rating,semantic_score,lexical_score,retrieval_score
0,shopee_25376324354,shopee,"Máy ảnh kĩ thuật số BẢN V2 48MEGAPIXELS - quay, chụp 48MP",NaN,VND,5.0,0.753583,0.0,0.800000
1,amazon_B002W1U1RW,amazon,Digital Concepts Digital Camera & Binocular 27379,89.99,USD,3.1,0.734914,1.0,0.788887
2,amazon_B000A7BIDQ,amazon,Digital Concepts 37mm TeleandWide Camcorder Lens Kit (KIT1337),25.99,USD,3.8,0.737665,0.5,0.736278
3,amazon_B007OXY8H2,amazon,Pentax SMC Pentax-DA L 18-55mm F3.5-5.6 AL (21827) for Digital SLR Cameras,76.00,USD,4.4,0.731679,0.5,0.703981
4,shopee_20681262132,shopee,"[Tặng thẻ nhớ] Máy ảnh kĩ thuật số digital mini camera v2 - quay, chụp 48MP, siêu mỏng nhỏ gọn",NaN,VND,5.0,0.717418,1.0,0.677944



QUERY: camera for photography


,product_id,source,product_name,price,currency,average_rating,semantic_score,lexical_score,retrieval_score
0,amazon_B07WFNWTV4,amazon,"BEIYANG Backdrop Stand, 7.5 FT x 10 FT Adjustable Photography Background Support System Kit with Carrying Bag for Photo Video Studio",39.98,USD,4.6,0.703087,0.5,0.908591
1,amazon_B08XVNSW72,amazon,"ISHARE Digital Camera for Photography with 2.8’’LCD, 20MP HD Photography Camera Rechargeable Point and Shoot Camera for Kids/Teenager/Seniors/Lear...",39.99,USD,3.7,0.693807,1.0,0.860807
2,amazon_B01BZ76Q54,amazon,"LimoStudio 24"" x 24"" / 60cm x 60cm Black & White Acrylic Reflective Table Top Display Background Boards for Photography Shooting, AGG1826",48.90,USD,4.1,0.700620,0.5,0.851597
3,amazon_B09SQ66VZN,amazon,Billingham Hadley Pro Canvas Bag for Camera - Parent Asin,276.00,USD,4.5,0.692460,0.5,0.775791
4,amazon_B0BNN99R3W,amazon,"ISHARE Digital Camera for Beginners - 30MP 1080P 18X Digital Zoom 2.8”LCD Screen, Point and Shoot Camera with 2X Batteries and 32G SD Card, Ideal ...",49.99,USD,4.0,0.685541,1.0,0.763319



QUERY: wireless headphones


,product_id,source,product_name,price,currency,average_rating,semantic_score,lexical_score,retrieval_score
0,amazon_B07LGY8V4B,amazon,Hermitshell Travel Case for WYZE Noise-Cancelling Headphones Wireless Over The Ear Bluetooth Headphones,15.99,USD,4.4,0.796609,1.0,0.939131
1,amazon_B0035GSGRK,amazon,Absolute Dynamic Wireless Headphones with RF Transmitter,21.82,USD,3.3,0.798270,1.0,0.921091
2,amazon_B09Q891GF1,amazon,"True Wireless Earbuds, TONSTEP Bluetooth 5.1 Headphones with Charging Case, Deep Bass Stereo Earphones, 30H Playtime, IPX6 Waterproof, Touch Contr...",44.99,USD,4.2,0.777533,1.0,0.754540
3,amazon_B0B2851KLD,amazon,"Wireless Earbuds Bluetooth Headphones with Wireless Charging Case, HI-FI Stereo in-Ear Headphones with Mic for Sports F9-36",24.27,USD,3.7,0.774778,1.0,0.730372
4,amazon_B0B33KCS1Q,amazon,"PHILIPS H4205 On-Ear Wireless Headphones with 32mm Drivers & BASS Boost on-Demand, Red",40.55,USD,4.0,0.768393,1.0,0.679122


# REVIEW SEMANTIC SEARCH


In [26]:
# ============================================================
# STEP 5 - CELL 20
# REVIEW SEMANTIC SEARCH
# ============================================================

def search_reviews(
    query,
    top_k=10,
    candidate_k=100,
    product_id=None,
    source=None,
    language=None,
    min_rating=None
):

    # Retrieve a wider global pool first.
    search_k = max(
        candidate_k,
        top_k
    )

    search_k = min(
        search_k,
        review_index.ntotal
    )


    query_vector = encode_query(
        query
    )


    scores, indices = (
        review_index.search(
            query_vector,
            search_k
        )
    )


    indices = indices[0]
    scores = scores[0]

    valid = indices >= 0

    indices = indices[valid]
    scores = scores[valid]


    result = (
        review_metadata
        .iloc[indices]
        .copy()
        .reset_index(drop=True)
    )

    result[
        "semantic_score"
    ] = scores


    if product_id is not None:

        result = result[
            result[
                "product_id"
            ].eq(product_id)
        ]


    if (
        source is not None
        and "source" in result.columns
    ):

        result = result[
            result["source"]
            .astype(str)
            .str.lower()
            .eq(
                str(source).lower()
            )
        ]


    if (
        language is not None
        and "language" in result.columns
    ):

        result = result[
            result["language"]
            .astype(str)
            .str.lower()
            .eq(
                str(language).lower()
            )
        ]


    if (
        min_rating is not None
        and "rating" in result.columns
    ):

        rating = pd.to_numeric(
            result[
                "rating"
            ],
            errors="coerce"
        )

        result = result[
            rating >= min_rating
        ]


    return (
        result
        .head(top_k)
        .reset_index(drop=True)
    )

# REVIEW RETRIEVAL TEST


In [27]:
# ============================================================
# STEP 5 - CELL 21
# REVIEW RETRIEVAL TEST
# ============================================================

review_queries = [

    "pin dùng được lâu",

    "battery lasts a long time",

    "chất lượng tốt",

    "easy to use"
]


for query in review_queries:

    print(
        "\n" + "=" * 70
    )

    print(
        "QUERY:",
        query
    )

    print(
        "=" * 70
    )

    result = search_reviews(
        query,
        top_k=5,
        candidate_k=100
    )

    cols = [

        "review_id",
        "product_id",
        "rating",
        "language",
        "source",
        "clean_review",
        "semantic_score"
    ]

    cols = [
        c
        for c in cols
        if c in result.columns
    ]

    display(
        result[cols]
    )


QUERY: pin dùng được lâu


,review_id,product_id,rating,language,source,clean_review,semantic_score
0,review_0343280,amazon_B0034Y1I6Y,5.0,en,amazon,long lasting battery,0.917324
1,review_0103350,amazon_B08TMJFSV3,5.0,en,amazon,Long lasting battery,0.916291
2,review_0456102,amazon_B015PQ3PKU,5.0,en,amazon,Battery fits as expected and lasts a long time!,0.915032
3,review_0100754,amazon_B01G8JO5F2,5.0,en,amazon,Long battery ✌🏽♥️,0.913037
4,review_0009295,amazon_B07Q25B9BJ,5.0,en,amazon,"Great battery, last a long time.",0.903225



QUERY: battery lasts a long time


,review_id,product_id,rating,language,source,clean_review,semantic_score
0,review_0343280,amazon_B0034Y1I6Y,5.0,en,amazon,long lasting battery,0.936162
1,review_0103350,amazon_B08TMJFSV3,5.0,en,amazon,Long lasting battery,0.934153
2,review_0109549,amazon_B01G8JO5F2,5.0,en,amazon,Battery last a lot,0.916450
3,review_0275857,amazon_B099FJB24J,5.0,en,amazon,Battery life lasts long,0.912155
4,review_0309725,amazon_B0935ZDCYD,4.0,en,amazon,Long battery life,0.911545



QUERY: chất lượng tốt


,review_id,product_id,rating,language,source,clean_review,semantic_score
0,review_0662664,amazon_B08PD7MBGH,5.0,en,amazon,Muy bueno calidad,0.970333
1,review_0343582,amazon_B00BV9TY8I,5.0,en,amazon,Bueno da calidad,0.966607
2,review_0718908,amazon_B07ZHYCRLQ,5.0,en,amazon,Muy buena calidad,0.964738
3,review_0703489,amazon_B0B2JX5N1C,5.0,en,amazon,Muy buena calidad,0.964738
4,review_0702540,amazon_B00OAXQ2LE,5.0,en,amazon,Muy buena calidad,0.964738



QUERY: easy to use


,review_id,product_id,rating,language,source,clean_review,semantic_score
0,review_0716248,amazon_B07BFCVJZC,5.0,en,amazon,easy to use,1.0
1,review_0698307,amazon_B006MCDUZM,5.0,en,amazon,easy to use,1.0
2,review_0684426,amazon_B07RT3KB3R,5.0,en,amazon,easy to use,1.0
3,review_0657613,amazon_B0075SUHKI,5.0,en,amazon,easy to use,1.0
4,review_0647850,amazon_B0BDF4DVXC,5.0,en,amazon,easy to use,1.0


# BUILD RAG METADATA


In [28]:
# ============================================================
# STEP 5 - CELL 22
# BUILD RAG METADATA
# ============================================================

print("=" * 70)
print("BUILD RAG METADATA")
print("=" * 70)


product_context_cols = [

    "product_id",
    "product_name",
    "category",
    "subcategory",
    "brand",

    "price",
    "currency",

    "average_rating",
    "review_count",
    "sold_count",

    "image_url",
    "product_url"
]


product_context_cols = [

    c
    for c in product_context_cols

    if c in product_metadata.columns
]


product_context = (
    product_metadata[
        product_context_cols
    ]
    .drop_duplicates(
        "product_id"
    )
)


rag_metadata = (
    review_metadata
    .merge(
        product_context,
        on="product_id",
        how="left"
    )
)


print(
    "RAG rows:",
    f"{len(rag_metadata):,}"
)

print(
    "Columns:",
    rag_metadata.columns.tolist()
)


assert (
    len(rag_metadata)
    == len(review_metadata)
)

print(
    "\n✅ RAG metadata aligned with review index"
)

BUILD RAG METADATA
RAG rows: 714,798
Columns: ['review_id', 'user_id', 'product_id', 'rating', 'timestamp', 'language', 'source', 'clean_review', 'review_embedding_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'image_url', 'product_url']

✅ RAG metadata aligned with review index


# BUILD RAG DOCUMENTS


In [29]:
# ============================================================
# STEP 5 - CELL 23
# BUILD RAG DOCUMENTS
# ============================================================

def safe_text(value):

    if pd.isna(value):
        return ""

    return str(value).strip()


def build_rag_document(row):

    parts = []


    name = safe_text(
        row.get(
            "product_name",
            ""
        )
    )

    if name:

        parts.append(
            f"Product: {name}"
        )


    category = safe_text(
        row.get(
            "category",
            ""
        )
    )

    if category:

        parts.append(
            f"Category: {category}"
        )


    brand = safe_text(
        row.get(
            "brand",
            ""
        )
    )

    if brand:

        parts.append(
            f"Brand: {brand}"
        )


    rating = row.get(
        "rating",
        None
    )

    if pd.notna(rating):

        parts.append(
            f"Review rating: {rating}/5"
        )


    review = safe_text(
        row.get(
            "clean_review",
            ""
        )
    )

    if review:

        parts.append(
            f"Review: {review}"
        )


    return " | ".join(
        parts
    )


rag_documents = (
    rag_metadata
    .copy()
)


rag_documents[
    "rag_document"
] = rag_documents.apply(
    build_rag_document,
    axis=1
)


rag_documents[
    "document_id"
] = np.arange(
    len(rag_documents)
)


print(
    "Documents:",
    f"{len(rag_documents):,}"
)

print(
    "Empty:",
    rag_documents[
        "rag_document"
    ].eq("").sum()
)


display(
    rag_documents[
        [
            "document_id",
            "product_id",
            "language",
            "rag_document"
        ]
    ].head(5)
)

Documents: 714,798
Empty: 0


,document_id,product_id,language,rag_document
0,0,amazon_B07N69T6TM,en,"Product: Toys for 4-5 Year Old Boys, Mom&myaboys 8 X 21 Kids Binoculars for Children,Compact Telescope Boys Gifts 4-8 Years Old to Bird Watching &..."
1,1,amazon_B01G8JO5F2,en,"Product: Senso Bluetooth Headphones, Best Wireless Sports Earbuds w/Mic IPX7 Waterproof HD Stereo Sweatproof Earphones for Gym Running Workout Noi..."
2,2,amazon_B07CJYMRWM,en,Product: Edifier P841 Comfortable Noise Isolating Over-Ear Headphones with Microphone and Volume Controls - White | Category: Cell Phones & Access...
3,3,amazon_B07CML419K,en,"Product: Fancii Small Personal Desk USB Fan, Portable Mini Table Fan with Twin Turbo Blades, Whisper Quiet Cyclone Air Technology - For Home, Offi..."
4,4,amazon_B09S6Y5BRG,en,"Product: Otium Bluetooth Earbuds Wireless Headphones Bluetooth Headphones, Sports Earbuds, IPX7 Waterproof Stereo Earphones for Gym Running 15 Hou..."


In [30]:
# ============================================================
# STEP 5 - CELL 24
# FINAL FAISS + RAG AUDIT
# ============================================================

print("=" * 70)
print("FINAL FAISS + RAG AUDIT")
print("=" * 70)


checks = {

    "product_index_rows":
        product_index.ntotal
        == len(product_metadata),

    "review_index_rows":
        review_index.ntotal
        == len(review_metadata),

    "product_dimension":
        product_index.d
        == DIM,

    "review_dimension":
        review_index.d
        == DIM,

    "rag_rows":
        len(rag_metadata)
        == len(review_metadata),

    "rag_document_rows":
        len(rag_documents)
        == len(review_metadata),

    "rag_documents_nonempty":
        rag_documents[
            "rag_document"
        ].ne("").all(),

    "document_id_order":
        np.array_equal(

            rag_documents[
                "document_id"
            ].values,

            np.arange(
                len(rag_documents)
            )
        )
}


for name, status in checks.items():

    print(
        ("✅" if status else "❌"),
        f"{name:<30}:",
        status
    )


print("\n" + "-" * 70)

print(
    "Product FAISS:",
    f"{product_index.ntotal:,}",
    "vectors"
)

print(
    "Review FAISS :",
    f"{review_index.ntotal:,}",
    "vectors"
)

print(
    "Dimension    :",
    DIM
)

print(
    "RAG documents:",
    f"{len(rag_documents):,}"
)


assert all(
    checks.values()
)

print(
    "\n✅ FAISS + RAG consistency checks passed"
)

FINAL FAISS + RAG AUDIT
✅ product_index_rows            : True
✅ review_index_rows             : True
✅ product_dimension             : True
✅ review_dimension              : True
✅ rag_rows                      : True
✅ rag_document_rows             : True
✅ rag_documents_nonempty        : True
✅ document_id_order             : True

----------------------------------------------------------------------
Product FAISS: 164,926 vectors
Review FAISS : 714,798 vectors
Dimension    : 384
RAG documents: 714,798

✅ FAISS + RAG consistency checks passed


# SAVE RAG ARTIFACTS


In [31]:
# ============================================================
# STEP 5 - CELL 25
# SAVE RAG ARTIFACTS
# ============================================================

print("=" * 70)
print("SAVE RAG ARTIFACTS")
print("=" * 70)


rag_metadata_path = (
    OUTPUT_DIR /
    "rag_metadata.csv"
)

rag_documents_path = (
    OUTPUT_DIR /
    "rag_documents.csv"
)


rag_metadata.to_csv(
    rag_metadata_path,
    index=False
)

rag_documents.to_csv(
    rag_documents_path,
    index=False
)


rag_config = {

    "embedding_model":
        MODEL_NAME,

    "embedding_dimension":
        DIM,

    "similarity":
        "cosine_via_normalized_inner_product",

    "product_index":
        "product_faiss.index",

    "review_index":
        "review_faiss.index",

    "product_vectors":
        int(
            product_index.ntotal
        ),

    "review_vectors":
        int(
            review_index.ntotal
        ),

    "product_candidate_k_default":
        200,

    "product_top_k_default":
        10,

    "review_candidate_k_default":
        100,

    "review_top_k_default":
        10,

    "product_reranking": {

        "semantic":
            0.70,

        "lexical":
            0.15,

        "rating":
            0.10,

        "popularity":
            0.05
    }
}


rag_config_path = (
    OUTPUT_DIR /
    "rag_config.json"
)


with open(
    rag_config_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        rag_config,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Saved:")
print(product_index_path)
print(review_index_path)
print(rag_metadata_path)
print(rag_documents_path)
print(rag_config_path)

SAVE RAG ARTIFACTS
Saved:
/kaggle/working/step5/product_faiss.index
/kaggle/working/step5/review_faiss.index
/kaggle/working/step5/rag_metadata.csv
/kaggle/working/step5/rag_documents.csv
/kaggle/working/step5/rag_config.json


In [32]:
# ============================================================
# STEP 5 - CELL 26
# OUTPUT SUMMARY
# ============================================================

print("=" * 70)
print("STEP 5 OUTPUT SUMMARY")
print("=" * 70)


output_files = [

    product_index_path,
    review_index_path,

    rag_metadata_path,
    rag_documents_path,

    rag_config_path
]


for path in output_files:

    if path.exists():

        size_mb = (
            path.stat().st_size
            / 1024**2
        )

        print(
            f"{path.name:<30}"
            f"{size_mb:>12.2f} MB"
        )


print("\n" + "=" * 70)
print("✅ STEP 5 COMPLETED")
print("=" * 70)

STEP 5 OUTPUT SUMMARY
product_faiss.index                 241.59 MB
review_faiss.index                 1047.07 MB
rag_metadata.csv                    388.13 MB
rag_documents.csv                   734.49 MB
rag_config.json                       0.00 MB

✅ STEP 5 COMPLETED
